<img src="icon.png" width=128/>

# Allo: Accelerator Design and Programming Language

# 2. Scheduling

In Allo, a `@kernel` says *what* to compute; a **`Schedule`** says *how* to map
that computation onto hardware. The two are deliberately decoupled: you write the
algorithm once, then apply schedule *primitives* (pipeline, unroll, partition,
tile, compose, stream, ...) to reshape its loop nest and memory layout for an
accelerator — without touching the math. This notebook is a tour of every
scheduling primitive, each with a short explanation and a runnable example
verified on the CPU backend or shown through emitted HLS C++.

*Series map:* **NB1 Frontend** (writing kernels) -> **NB2 Scheduling** (this
notebook) -> **NB3 Simulation** (running/verifying) -> **NB4 Backends** (CPU /
Vitis codegen & synthesis).

In [ ]:
import numpy as np
from allo.lang import kernel, range, i32, f32
from allo.schedule.errors import ScheduleLookupError, ConsumedHandleError

## 1. Why a schedule? Algorithm vs. mapping

A kernel is a pure description of a computation. `kernel.schedule()` compiles it
and hands back a `Schedule` bound to the kernel's MLIR module. From there you
*select* loops/buffers by name and apply primitives that rewrite the module.

The classic "hello world" of scheduling: take a flat 16-element loop, **split**
it into an outer loop over 4-element tiles and an inner loop within a tile, then
**pipeline** the inner loop so a new iteration launches every cycle (`ii=1`).
None of this changes the result — only how the hardware executes it.

> A templated kernel (`@kernel(T, N) def gemm(...)`) must be *specialized* first,
> e.g. `gemm[i32, 32].schedule()`. Loops are selected by their **iterator name**
> (`for i in range(16)` → `"i"`) — no `name=` needed; see section 3.

In [ ]:
@kernel
def add1(A: i32[16], B: i32[16]):
    for i in range(16):
        B[i] = A[i] + 1

s = add1.schedule()
outer, inner = s.split("i", factor=4)   # select loop "i" by its iterator name
s.pipeline(inner, ii=1)                 # pipeline the inner loop, II=1

# The pipeline request is recorded as an attribute in the payload MLIR module.
for line in str(s).splitlines():
    if "pipeline.ii" in line:
        print(line.strip())

## 2. The schedule model: payload, snapshot, transform script

A `Schedule` has three cooperating pieces:

| Piece                | Role                                                                 |
| -------------------- | -------------------------------------------------------------------- |
| **payload**          | the MLIR module being transformed                                    |
| **snapshot**         | an immutable index of ops/buffers (keyed by stable IDs); selection reads it |
| **transform script** | accumulated MLIR *transform-dialect* ops, run against the payload by `apply()` |

Primitives come in two flavours:

- **Tagging primitives** (`pipeline`, `unroll(tag_only=True)`, `partition`,
  `dataflow`, `bind_storage`, `cse/dce/...`) attach attributes without changing
  loop topology. They **defer**: they append to the script and return `s` for
  chaining, and only take effect on `apply()`. Refs you captured earlier stay
  valid.
- **Structural primitives** (`split`, `reorder`, `tile`, `flatten`, `compute_at`,
  `buffer_at`, `reuse_at`, `outline`) change the nest/function structure. They
  **apply immediately**, rebuild the snapshot, and return fresh refs; refs from
  before the call go stale.

`s.dirty` tells you whether tags are pending. Reading `s.payload` or
`s.snapshot` while dirty auto-applies them. You can batch several tags and pay a
single `apply()`.

In [ ]:
@kernel
def batched(A: i32[16], B: i32[16]):
    for i in range(16):
        B[i] = A[i] * 3

s = batched.schedule()
print("dirty at start   :", s.dirty)        # nothing queued yet
s.pipeline("i", ii=2)                        # tagging primitive -> deferred
print("dirty after tag  :", s.dirty)         # True: pending
s.apply()                                    # run the batch
print("dirty after apply:", s.dirty)         # False

# Reading `payload` while dirty applies automatically:
s2 = batched.schedule()
s2.pipeline("i", ii=4)
_ = str(s2.payload)                          # triggers apply()
print("auto-applied on payload read, dirty:", s2.dirty)

## 3. Selecting targets

Primitives operate on **refs**. The short aliases resolve one ref (or a tuple):

| Alias                    | Result                            |
| ------------------------ | --------------------------------- |
| `s.loop(name)`           | one `LoopRef`                     |
| `s.loops(*names)`        | tuple of `LoopRef` (all if empty) |
| `s.buffer(name)`         | one `BufferRef`                   |
| `s.op(name, kind=...)`   | one `OpRef`                       |

**A loop's name is its iterator.** `for i in range(...)` is selectable as `"i"`
with no `name=` — that is the recommended style. Reach for
`range(..., name="...")` only when a kernel has **two loops with the same
iterator name** (otherwise selection raises `AmbiguousLookupError`), or when you
want a stable name independent of the variable. Buffer names are the kernel
argument / local-variable names.

**String sugar.** A primitive whose target *kind* is unambiguous accepts a bare
name instead of a ref — the scheduler resolves it to the right kind:

| Sugar                              | Equivalent to                                        |
| ---------------------------------- | ---------------------------------------------------- |
| `s.pipeline("i")`                  | `s.pipeline(s.loop("i"))`                             |
| `s.unroll("i")` / `s.split("i")`   | `s.unroll(s.loop("i"))` / `s.split(s.loop("i"))`     |
| `s.reorder(("k", "j"))`            | `s.reorder((s.loop("k"), s.loop("j")))`              |
| `s.partition("A", ...)`            | `s.partition(s.buffer("A"), ...)`                    |
| `s.reuse_at("A", "x")`             | `s.reuse_at(s.buffer("A"), s.loop("x"))`             |

`s.pipeline("i")` only ever looks for a *loop* named `i`, never a buffer, so the
explicit `s.loop("i")` is redundant. You still call `s.loop(...)` / `s.buffer(...)`
when you need to **hold a ref** — to reuse it later, or hand it to `s.live()`.
For anything finer, drop to `s.query.loop/op/buffer(...)`, which returns a
*selection* you finish with `.one()`, `.first()`, `.all()`, or `.names(*names)`;
`under=` scopes a lookup beneath another op.

In [ ]:
@kernel
def gemm(A: f32[8, 8], B: f32[8, 8], C: f32[8, 8]):
    for i in range(8):
        for j in range(8):
            for k in range(8):
                C[i, j] += A[i, k] * B[k, j]

s = gemm.schedule()

# Loops are named after their iterators -- no `name=` was needed:
print("one loop  :", s.loop("k").key)
print("two loops :", [l.key for l in s.loops("i", "j")])
print("all loops :", [l.key for l in s.loops()])

# A buffer by name (its key points at the owning arg/result):
print("buffer C  :", s.buffer("C").key)

# Low-level query: every affine.store op, or just the first.
stores = s.query.op(kind="affine.store").all()
print("num stores:", len(stores), "| first store key:", stores[0].key)

## 4. Generic cleanup passes

`cse`, `dce`, `licm`, and `canonicalize` run the corresponding MLIR passes over
their target (defaulting to the primary function). They are tagging primitives —
chain them and call `.apply()`. They never change results; they tidy the IR the
other primitives produce.

`licm` (loop-invariant code motion) hoists work out of a loop, so target a
**loop** with it rather than the whole function.

In [ ]:
@kernel
def clean(A: i32[8, 8], B: i32[8, 8]):
    for i in range(8):
        for j in range(8):
            B[i, j] = A[i, j] + 1

s = clean.schedule()
s.cse().dce().canonicalize()        # chained cleanup over the function
print("cleanup verifies :", s.payload.operation.verify())

s2 = clean.schedule()
s2.licm("i")                     # hoist invariants out of loop i
print("licm verifies    :", s2.payload.operation.verify())

## 5. Loop & memory tags (the HLS pragmas)

These tagging primitives annotate the IR for the hardware backend. The Vitis
emitter turns each attribute into an `#pragma HLS ...`. We don't need a toolchain
to see them — `s.export("vitis").hls_code` returns the generated C++ as a string
(full backend detail is NB4).

- `pipeline(loop, ii=)` -> `#pragma HLS pipeline II=`
- `partition(buffer, dim=, kind=, factor=)` -> `#pragma HLS array_partition`
  (`kind` is `s.Complete` with `factor=0`, or `s.Block`/`s.Cyclic` with
  `factor>0`; `dim=0` = all dims)
- `unroll(loop, factor=, tag_only=)` — see below
- `dataflow()` -> `#pragma HLS dataflow` (task-level parallelism)
- `bind_storage(buffer, impl=, mem_type=)` -> `#pragma HLS bind_storage`
  (a Vitis memory-resource hint; the CPU backend ignores it)

In [ ]:
@kernel
def mm(A: f32[8, 8], B: f32[8, 8], C: f32[8, 8]):
    for i in range(8):
        for j in range(8):
            for k in range(8):
                C[i, j] += A[i, k] * B[k, j]

s = mm.schedule()
s.pipeline("j", ii=1)                             # II=1 pipeline
s.partition("A", dim=2, kind=s.Cyclic, factor=2)  # bank A along dim 2
code_hls = s.export("vitis").hls_code

for l in code_hls.splitlines():
    p = l.strip()
    if "#pragma HLS" in p and ("pipeline" in p or "array_partition" in p):
        print(p)

### `unroll`: physical vs. attribute

`unroll(loop, factor=0)` **physically** replicates the loop body immediately
(`factor=0` = full unroll), exposing parallel operators (an adder tree, several
MACs). `unroll(loop, factor=, tag_only=True)` instead only *tags* the loop with
`#pragma HLS unroll` and defers, leaving the loop intact in the IR.

In [ ]:
# tag_only: the loop stays, an unroll pragma is emitted
@kernel
def uu_tag(A: i32[8], B: i32[8]):
    for i in range(8):
        B[i] = A[i] + 1

s_tag = uu_tag.schedule()
s_tag.unroll("i", factor=4, tag_only=True).apply()
print("tag_only -> attribute in IR :", "unroll.f = 4" in str(s_tag.payload))
print("tag_only -> HLS pragma      :",
      "#pragma HLS unroll" in s_tag.export("vitis").hls_code)

# physical: the loop is gone, its body replicated 8x (a fresh kernel copy)
@kernel
def uu_phys(A: i32[8], B: i32[8]):
    for i in range(8):
        B[i] = A[i] + 1

s_phys = uu_phys.schedule()
print("before: affine.for count    :", str(s_phys.payload).count("affine.for"))
s_phys.unroll("i", factor=0)                # full physical unroll
s_phys.apply()
print("after : affine.for count    :", str(s_phys.payload).count("affine.for"),
      "| addi ops:", str(s_phys.payload).count("arith.addi"))

### `bind_storage`

Pin a buffer to a specific on-chip memory resource. `impl` picks the primitive
(`s.BRAM`, `s.URAM`, `s.LUTRAM`, ...) and `mem_type` the port configuration
(`s.RAM_1P`, `s.RAM_2P`, `s.RAM_S2P`, ...).

In [ ]:
@kernel
def staged(A: f32[16], B: f32[16]):
    buf: f32[16]
    for i in range(16):
        buf[i] = A[i] + 1.0
    for j in range(16):
        B[j] = buf[j]

s = staged.schedule()
s.bind_storage("buf", impl=s.BRAM, mem_type=s.RAM_2P)
for l in s.export("vitis").hls_code.splitlines():
    if "bind_storage" in l:
        print(l.strip())

## 6. Loop restructuring (structural primitives)

These reshape the loop nest itself. Each applies immediately and returns fresh
refs; a ref you held before the call is stale afterward. Every example below runs
on the CPU backend and is checked against a NumPy reference — restructuring must
preserve results, it only changes locality / pipelinability. Runs below use the shorthand `s("cpu", *args)` — sugar for `s.export("cpu")(*args)` that calls a backend simulation directly (full backend detail is NB3/NB4).

- `split(loop, factor=)` -> `(outer, inner)` (loop `i` becomes `i.outer`/`i.inner`).
- `reorder(loops)` permutes a perfectly-nested affine band (>=2 loops).
- `tile(loops, factors=)` -> `(tile_loops, point_loops)` = split each axis + reorder.
- `flatten(loops)` collapses a perfect nest into one loop.

The `reorder` example reads the physical nest order out of the emitted HLS C++,
since `s.loops()` reports a canonical (not nest) order.

### `split`

In [ ]:
@kernel
def v(A: i32[16], B: i32[16]):
    for i in range(16):
        B[i] = A[i] + 1

# split: i -> i.outer / i.inner
s = v.schedule()
outer, inner = s.split("i", factor=4)
print("split -> new loops :", [l.key for l in s.loops()])
A = np.arange(16, dtype=np.int32); B = np.zeros(16, dtype=np.int32)
s("cpu", A, B)
print("split result ok    :", np.array_equal(B, A + 1))
print(s)

### `reorder` — change the nest order, keep the result

In [ ]:
import re

@kernel
def gemm(A: f32[8, 8], B: f32[8, 8], C: f32[8, 8]):
    for i in range(8):
        for j in range(8):
            for k in range(8):
                C[i, j] += A[i, k] * B[k, j]

def nest_order(hls):
    """Loop nesting order as it appears in the emitted HLS C++."""
    return re.findall(r"loop_([A-Za-z0-9]+)_[A-Za-z0-9_]*: for", hls)

s = gemm.schedule()
# the kernel is written i, j, k; reorder swaps the inner two -> i, k, j
s.reorder(("k", "j"))
print("nest after reorder :", nest_order(s.export("vitis").hls_code))   # i, k, j

# The same schedule verifies on the CPU backend (reorder preserves the result):
A = np.random.rand(8, 8).astype(np.float32)
B = np.random.rand(8, 8).astype(np.float32)
C = np.zeros((8, 8), dtype=np.float32)
s("cpu", A, B, C)
print("reorder preserves A@B:", np.allclose(C, A @ B, rtol=1e-4))
print(s)

### `tile` and `flatten`

In [ ]:
# tile: outer "tile" loops + inner "point" loops per axis
@kernel
def t_tile(A: i32[8, 8], B: i32[8, 8], C: i32[8, 8]):
    for i in range(8):
        for j in range(8):
            C[i, j] = A[i, j] + B[i, j]

s = t_tile.schedule()
tiles, points = s.tile(("i", "j"), factors=[2, 4])
print("tile loops :", [l.key for l in s.loops()])
A = np.random.randint(0, 10, (8, 8)).astype(np.int32)
B = np.random.randint(0, 10, (8, 8)).astype(np.int32)
C = np.zeros((8, 8), dtype=np.int32)
s("cpu", A, B, C)
print("tile result ok :", np.array_equal(C, A + B))
print(s)

# flatten: two perfectly-nested loops collapse into one (a fresh kernel copy)
@kernel
def t_flat(A: i32[8, 8], B: i32[8, 8], C: i32[8, 8]):
    for i in range(8):
        for j in range(8):
            C[i, j] = A[i, j] + B[i, j]

s = t_flat.schedule()
print("affine.for before flatten:", str(s.payload).count("affine.for"))
one = s.flatten(("i", "j"))
print("affine.for after  flatten:", str(s.payload).count("affine.for"),
      "| flat loop:", one.key)
C2 = np.zeros((8, 8), dtype=np.int32)
s("cpu", A, B, C2)
print("flatten result ok :", np.array_equal(C2, A + B))
print(s)

## 7. Data movement & localization

These structural primitives move computation on-chip or introduce staging
buffers.

- `compute_at(producer, axis)` fuses a producer loop nest into a consumer loop at
  the given axis, shortening the producer's live range (loop fusion).
- `buffer_at(buffer, axis)` stages a buffer local to `axis` (a `{base}.local`
  allocation) — copy-in for reads, copy-out for writes.
- `reuse_at(buffer, axis, ring=False)` builds the classic **line/window buffer**
  for stencils/convolutions: it keeps only the small sliding window live across
  iterations instead of the whole array (`{base}.reuse`). `ring=True` uses a
  circular buffer instead of physically shifting.

In [ ]:
# compute_at: fuse two elementwise bands into one loop
@kernel
def two_band(A: i32[8, 8], C: i32[8, 8]):
    B: i32[8, 8] = 0
    for bi in range(8):
        for bj in range(8):
            B[bi, bj] = A[bi, bj] + 1
    for ci in range(8):
        for cj in range(8):
            C[ci, cj] = B[ci, cj] * 2

s = two_band.schedule()
s.compute_at("bi", "ci")                     # move producer into consumer's ci
A = np.random.randint(0, 10, (8, 8)).astype(np.int32)
C = np.zeros((8, 8), dtype=np.int32)
s("cpu", A, C)
print("compute_at result == (A+1)*2 :", np.array_equal(C, (A + 1) * 2))
print(s)

In [ ]:
# buffer_at: stage output B on-chip per row of the outer loop
@kernel
def addone(A: f32[8, 8], B: f32[8, 8]):
    for i in range(8):
        for j in range(8):
            B[i, j] = A[i, j] + 1.0

s = addone.schedule()
local = s.buffer_at("B", "i")
print("buffer_at created  :", local.key)     # B.local
A = np.random.rand(8, 8).astype(np.float32); B = np.zeros((8, 8), dtype=np.float32)
s("cpu", A, B)
print("buffer_at result ok:", np.allclose(B, A + 1.0, rtol=1e-5))
print(s)

In [ ]:
# reuse_at: a 3-tap horizontal blur -> sliding line buffer over x
H, W = 10, 10

@kernel
def blur(A: i32[H, W], B: i32[H, 8]):
    for y in range(H):
        for x in range(8):
            B[y, x] = A[y, x] + A[y, x + 1] + A[y, x + 2]

s = blur.schedule()
rb = s.reuse_at("A", "x")                     # window buffer along x
print("reuse buffer       :", rb.key)         # A.reuse
A = np.random.randint(0, 10, (H, W)).astype(np.int32)
B = np.zeros((H, 8), dtype=np.int32)
s("cpu", A, B)
ref = A[:, 0:8] + A[:, 1:9] + A[:, 2:10]       # numpy blur reference
print("reuse_at blur ok   :", np.array_equal(B, ref))
print(s)

# ring=True: same result, circular buffer instead of a physical shift (fresh copy)
@kernel
def blur_ring(A: i32[H, W], B: i32[H, 8]):
    for y in range(H):
        for x in range(8):
            B[y, x] = A[y, x] + A[y, x + 1] + A[y, x + 2]

s = blur_ring.schedule()
s.reuse_at("A", "x", ring=True)
B2 = np.zeros((H, 8), dtype=np.int32)
s("cpu", A, B2)
print("reuse_at ring ok   :", np.array_equal(B2, ref))
print(s)

## 8. Outlining a loop into a function

`outline(target, func_name=, mapping=None)` extracts an operation (e.g. a loop)
into its own function so it can be reused or scheduled independently, returning
`(kernel_ref, call_ref)` — a ref to the new function and a ref to the call site
that replaces the loop.

- With `mapping=None` it emits a plain `func.func` / `func.call` pair.
- When `mapping` is an int or a sequence of positive ints it emits the spatial
  `allo.kernel` / `allo.invoke` PE form — the same shape an inline
  `@kernel(mapping=...)` produces (see NB1's spatial-mapping section). The
  `mapping` gives the PE-array dimensions.

```python
s = scale.schedule()
# Lift the outer loop into its own function @stage0, replacing it with a call:
kernel_ref, call_ref = s.outline("i", func_name="stage0")
#   kernel_ref.key -> "stage0"     (the new func.func)
#   call_ref.key   -> "i.call"     (the func.call that took the loop's place)

# Spatial form: a 2x1 PE array instead of a plain function.
producer, consumer = s.affine(("i", "ci"))
axis = s.compute_at(producer, consumer)
outer, inner = s.split(axis, factor=4)
stage, call = s.outline(inner, func_name="pe", mapping=[2, 1])
```

> **Build note.** `outline` is functional as a queued transform in this checkout
> but its *materialization* is unstable (it can fail verification), so this
> section is illustrative rather than executed — every other primitive in this
> notebook runs live. Prefer expressing PE arrays directly with
> `@kernel(mapping=...)` (NB1) until this stabilizes.

## 9. Composing scheduled kernels

Real designs are built from stages. Allo schedules each kernel **independently**,
then stitches them together: when a top kernel invokes a sub-kernel, the compiler
specializes a private copy named `"{primary}.{callee}"`. `s.compose(*callees)`
replays each direct callee's *entire* schedule onto that copy.

Below, a GEMM stage and an add-one stage are scheduled separately (each pipelines
its `j` loop), then composed into a two-stage `top`. Composition is variadic
(`compose(a, b)` == `compose(a); compose(b)`) and transitive.

In [ ]:
M = K = N = 8

@kernel
def gemm(A: i32[M, K], B: i32[K, N], C: i32[M, N]):
    for i in range(M):
        for j in range(N):
            for k in range(K):
                C[i, j] += A[i, k] * B[k, j]

@kernel
def addone(C: i32[M, N], D: i32[M, N]):
    for i in range(M):
        for j in range(N):
            D[i, j] = C[i, j] + 1

@kernel
def top(A: i32[M, K], B: i32[K, N], C: i32[M, N], D: i32[M, N]):
    gemm(A, B, C)                             # stage 1
    addone(C, D)                              # stage 2

gs = gemm.schedule();    gs.pipeline("j", ii=1)
as_ = addone.schedule(); as_.pipeline("j", ii=1)

ts = top.schedule()
ts.compose(gs, as_)                           # replay both stage schedules
print("private copies present:", "@top.gemm" in str(ts) and "@top.addone" in str(ts))

A = np.random.randint(0, 10, (M, K)).astype(np.int32)
B = np.random.randint(0, 10, (K, N)).astype(np.int32)
C = np.zeros((M, N), dtype=np.int32); D = np.zeros((M, N), dtype=np.int32)
ts("cpu", A, B, C, D)
print("composed result == A@B + 1:", np.array_equal(D, A @ B + 1))
print(ts)

## 10. Streaming: DRAM boundaries -> on-chip FIFOs

When one stage writes an intermediate that the next stage reads, that array would
normally round-trip through DRAM. `s.streamline(producer, consumer, ...)` fuses
the boundary into an on-chip stream so the stages run as a producer/consumer
dataflow pair:

- **one -> one**: a FIFO
- **one -> many**: a generated `tee` (residual / skip connections)
- **many -> one**: a generated `merge` (each producer fills a disjoint block)

Two knobs matter: **`lanes`** widens the boundary to *L* parallel FIFOs moving
*L* elements/cycle (the bandwidth lever; the contiguous dim must divide by it),
and **`depth`** is the FIFO depth — on a reconvergent fork/join the short branch's
FIFO must hold the latency skew or the dataflow **deadlocks** (`streamline` warns
and names the depth to set). Follow `streamline` with `dataflow()` to run the
stages concurrently.

Here a 3-stage chain `s1 -> s2 -> s3` has *both* intermediates streamlined; the
middle stage becomes stream-in **and** stream-out.

In [ ]:
N = 16

@kernel
def s1(X: f32[N, N], T1: f32[N, N]):
    for i in range(N):
        for j in range(N):
            T1[i, j] = X[i, j] + 1.0

@kernel
def s2(T1: f32[N, N], T2: f32[N, N]):
    for i in range(N):
        for j in range(N):
            T2[i, j] = T1[i, j] * 2.0

@kernel
def s3(T2: f32[N, N], O: f32[N, N]):
    for i in range(N):
        for j in range(N):
            O[i, j] = T2[i, j] + 3.0

@kernel
def chain(X: f32[N, N], O: f32[N, N]):
    T1: f32[N, N]
    T2: f32[N, N]
    s1(X, T1); s2(T1, T2); s3(T2, O)

cs = chain.schedule()
cs.streamline("s1", "s2")                     # DRAM T1 -> FIFO
cs.streamline("s2", "s3", lanes=1, depth=2)   # DRAM T2 -> FIFO
cs.dataflow()                                 # run the three stages concurrently
print("boundary converted to stream:", "allo.stream" in str(cs))

X = np.random.rand(N, N).astype(np.float32); O = np.zeros((N, N), dtype=np.float32)
cs("cpu", X, O)
print("streamed result == (X+1)*2 + 3:", np.allclose(O, (X + 1.0) * 2.0 + 3.0, rtol=1e-4))

## 11. Ref lifetime

A structural primitive *consumes* the refs it rewrites. `split` turns loop `i`
into `i.outer`/`i.inner`, so the original `i` ref is dead — reusing it raises
`ConsumedHandleError`. Recover a live ref three ways: use the refs the primitive
**returned**, **re-select** by the new name, or `s.live(old_ref)` to rebind a
sibling ref whose schedule ID still exists.

In [ ]:
@kernel
def lif(A: i32[16], B: i32[16]):
    for i in range(16):
        B[i] = A[i] + 1

s = lif.schedule()
i = s.loop("i")                               # hold a ref (needed to show it dies)
outer, inner = s.split(i, factor=4)           # `i` is now consumed

try:
    s.pipeline(i, ii=1)                        # reusing the dead ref -> error
except ConsumedHandleError:
    print("reusing split-away `i` raised ConsumedHandleError")

# Recover via the returned ref (or re-select "i.inner"):
s.pipeline(inner, ii=1).apply()
print("recovered; loops now:", [l.key for l in s.loops()])

In [ ]:
# s.live(): rebind a *sibling* ref that went stale when another loop changed.
@kernel
def lif2(A: i32[8, 8], B: i32[8, 8], C: i32[8, 8]):
    for i in range(8):
        for j in range(8):
            C[i, j] = A[i, j] + B[i, j]

s = lif2.schedule()
i, j = s.loops("i", "j")
outer, inner = s.split(i, factor=4)           # `j` captured from a prior state
j = s.live(j)                                 # rebind before reuse
s.pipeline(j, ii=1).apply()
print("s.live() rebind ok; loops:", [l.key for l in s.loops()])

## 12. Debugging & error types

Inspection helpers:

| Helper                       | Effect                                          |
| ---------------------------- | ----------------------------------------------- |
| `s.format_tree()`            | text tree of the current snapshot (returns str) |
| `s.dump_tree()`              | prints and returns that tree                    |
| `s.dump_transform_script()`  | the pending transform script as MLIR text       |
| `s.debug_dump()`             | dirty state, op/value counts, tree, and script  |

Schedule errors are source-aware and point at the Python call site:
`ScheduleLookupError` (name/path doesn't resolve), `AmbiguousLookupError` (a
single-target lookup matches many), `ConsumedHandleError` (reusing a consumed
ref), `InvalidScheduleArgumentError` (out-of-range argument), and
`ScheduleTransformError` (the transform or its result fails verification).

In [ ]:
@kernel
def dbg(A: i32[8, 8], B: i32[8, 8]):
    for i in range(8):
        for j in range(8):
            B[i, j] = A[i, j] + 1

s = dbg.schedule()
print(s.format_tree(include_values=False))

# A missing loop name is a source-aware ScheduleLookupError:
try:
    s.loop("does_not_exist")
except ScheduleLookupError:
    print("selecting an unknown loop raised ScheduleLookupError")

## 13. Capstone: the RMSNorm schedule

Finally, a realistic schedule from the Allo transformer library
(`allo/library/transformer/rms.py`). RMSNorm normalizes each row of `x[S, D]`:
`y = x * rsqrt(mean(x^2) + eps) * g`. Each row needs two passes (sum-of-squares,
then normalize), so `SB` rows are staged on-chip and read once; `L` lanes are
processed per cycle.

The schedule combines primitives we've met:

- **`partition`** the on-chip buffers so `L` lanes are readable in one cycle:
  `buf` is banked `Cyclic` by `L` (adjacent hidden elements land in different
  banks), and the tiny per-row `ss`/`inv` vectors are split `Complete` into
  registers.
- **physical `unroll`** of the `L` lane loop (`rl`/`nl`) turns the per-lane
  multiplies into a parallel **adder tree** — `L` MACs/cycle.
- **`flatten` + `pipeline`** the `(row-chunk, row)` nest into one continuous
  pipeline with the row index fast-varying. The sum-of-squares is a loop-carried
  float add whose recurrence can't hit `II=1`; interleaving `SB` rows overlaps
  that latency across independent rows.

The library wraps `(kernel, schedule)` in a `Module` subclass (see
`allo.lang.Module`); here we build the kernel + schedule directly.

> **Why `name=` here?** Both lane loops iterate `l` (pass 1 `rl`, pass 2 `nl`),
> so their iterator names collide — and the schedule needs stable handles to
> `flatten` / `pipeline` / `unroll` specific loops. Naming the loops is exactly
> the "disambiguate duplicate iterators" case from section 3.

In [ ]:
from allo.operators import math as m

def make_rmsnorm(Tin, Tacc, Tout, S, D, L=16, SB=8, eps=1e-5, ii=1):
    DT = D // L
    SBN = S // SB
    invD = 1.0 / D

    @kernel
    def rmsnorm(x: Tin[S, D], g: Tin[D], y: Tout[S, D]):
        for sb in range(SBN):
            buf: Tacc[SB, D]                  # SB rows staged on-chip, read once
            ss: Tacc[SB]                      # per-row sum of squares
            for s0 in range(SB):
                ss[s0] = 0.0
            # pass 1: read SB rows + reduce (row fast-varying hides the fadd recurrence)
            for ct in range(DT):
                for s in range(SB):
                    part: Tacc = 0.0
                    for l in range(L, name="rl"):    # unrolled -> adder tree
                        xv: Tacc = x[sb * SB + s, ct * L + l]
                        buf[s, ct * L + l] = xv
                        part = part + xv * xv
                    ss[s] = ss[s] + part
            inv: Tacc[SB]
            for s2 in range(SB):
                inv[s2] = m.rsqrt(ss[s2] * invD + eps)
            # pass 2: normalize, L lanes/cycle, flattened pipeline
            for s3 in range(SB):
                for dt in range(DT):
                    for l in range(L, name="nl"):    # unrolled
                        o: Tacc = buf[s3, dt * L + l] * inv[s3] * g[dt * L + l]
                        y[sb * SB + s3, dt * L + l] = o

    s = rmsnorm.schedule()
    s.partition("buf", dim=2, kind=s.Cyclic, factor=L)   # L lanes/cycle
    s.partition("ss", dim=1, kind=s.Complete)            # registers
    s.partition("inv", dim=1, kind=s.Complete)
    s.unroll("s0")
    s.unroll("rl")                                # adder tree over L lanes
    s.pipeline(s.flatten(("ct", "s")), ii=ii)   # one continuous pass-1 pipeline
    s.unroll("s2")
    s.unroll("nl")
    s.pipeline(s.flatten(("s3", "dt")), ii=ii)   # one continuous pass-2 pipeline
    return rmsnorm, s

In [ ]:
S, D, L, SB = 16, 64, 16, 8
k, s = make_rmsnorm(f32, f32, f32, S, D, L, SB)

# Inspect the emitted HLS pragmas this schedule produces:
hls = s.export("vitis").hls_code
print("pipeline II=1 pragma :", "#pragma HLS pipeline II=1" in hls)
print("cyclic partition     :", "cyclic" in hls.lower())

# Verify functionally against a NumPy RMSNorm reference (same schedule):
x = np.random.rand(S, D).astype(np.float32)
g = np.random.rand(D).astype(np.float32)
y = np.zeros((S, D), dtype=np.float32)
s("cpu", x, g, y)
ref = x * (1.0 / np.sqrt(np.mean(x ** 2, axis=1, keepdims=True) + 1e-5)) * g
print("RMSNorm matches NumPy:", np.allclose(y, ref, rtol=1e-3, atol=1e-4))
print(s.export("vitis").hls_code)

### Recap

You've now seen every schedule primitive: selection (`loop`/`loops`/`buffer`/
`query`), generic passes, loop & memory tags (`pipeline`, `unroll`, `partition`,
`dataflow`, `bind_storage`), loop restructuring (`split`, `reorder`,
`tile`, `flatten`), localization (`compute_at`, `buffer_at`, `reuse_at`),
`outline`, `compose`, and `streamline` — plus the schedule model (deferred tags
vs. immediate structural changes), ref lifetime, and debugging. Next up, **NB3
Simulation** runs and verifies these designs, and **NB4 Backends** takes them to
CPU and Vitis HLS.